# LLM-based classifier

Install requirements

In [ ]:
# !pip3 install -q transformers torch accelerate sentencepiece

Initialization

In [1]:
import os
import re
import torch
import pandas as pd
from typing import List, Dict, Tuple
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/negin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# # =========================================================
# # Configuration
# # =========================================================
# MODEL_NAME = "google/flan-t5-small"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MAX_INPUT_LENGTH = 512
# MAX_NEW_TOKENS = 10


# # =========================================================
# # Load Data
# # =========================================================
# train_df = pd.read_csv('data/6/train.csv')

# # =========================================================
# # Prompt Templates
# # =========================================================
# PROMPT_TEMPLATES = {
#     "prompt_1": """
#     You are a text classification system.

#     Classify the sentiment of the following text into exactly one label from:
#     {labels}

#     Return only one label.

#     Text:
#     {text}

#     Label:
#     """.strip(),

#     "prompt_2": """
#     Classify the sentiment of this text as one of these labels: {labels}.

#     Text: {text}

#     Answer with only one label.
#     """.strip(),

#         "prompt_3": """
#     Read the text and predict its sentiment.

#     Possible labels: {labels}

#     Text:
#     {text}

#     Output only the correct label.
#     """.strip(),

#     "prompt_4": """
#     You must perform sentiment classification.

#     Allowed labels:
#     {labels}

#     Rules:
#     - Choose one label only
#     - Do not explain

#     Text:
#     {text}

#     Sentiment:
#     """.strip(),

#     "prompt_5": """
#     Determine whether the sentiment of the following text is:
#     {labels}

#     Text:
#     {text}

#     Return only the label.
#     """.strip(),
#     }



Functions

In [2]:
def normalize_label(raw_output: str, valid_labels: List[str]) -> str:
    """
    Convert raw model output into one valid label.

    Strategy:
    1. Strip spaces and lowercase the output
    2. Return exact match if available
    3. Otherwise, search for the first valid label inside the output
    4. Return 'UNKNOWN' if no valid label is found
    """
    cleaned_output = str(raw_output).strip().lower()
    normalized_labels = [str(label).strip().lower() for label in valid_labels]

    if cleaned_output in normalized_labels:
        return cleaned_output

    for label in normalized_labels:
        if re.search(rf"\b{re.escape(label)}\b", cleaned_output):
            return label

    return "UNKNOWN"


def load_hf_model_and_tokenizer(
    model_name: str,
    device: str
):
    """
    Load the appropriate Hugging Face tokenizer and model.

    Uses:
    - AutoModelForSeq2SeqLM for T5 / FLAN-T5 style models
    - AutoModelForCausalLM for GPT-style models

    Returns:
        tokenizer, model, model_type
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    lowered_name = model_name.lower()
    is_seq2seq = any(name_part in lowered_name for name_part in ["t5", "flan"])

    model_dtype = torch.float16 if device == "cuda" else torch.float32

    if is_seq2seq:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=model_dtype
        )
        model_type = "seq2seq"
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=model_dtype
        )
        model_type = "causal"

    model.to(device)
    model.eval()

    return tokenizer, model, model_type


def build_prompt(
    prompt_template: str,
    text: str,
    valid_labels: List[str]
) -> str:
    """
    Format one prompt using the provided template.

    Expected placeholders in prompt_template:
    - {text}
    - {labels}
    """
    labels_text = ", ".join([str(label) for label in valid_labels])
    return prompt_template.format(text=str(text), labels=labels_text)


def predict_single_text(
    text: str,
    prompt_template: str,
    valid_labels: List[str],
    tokenizer,
    model,
    model_type: str,
    max_input_length: int,
    max_new_tokens: int,
    device: str
) -> Tuple[str, str]:
    """
    Predict one label for one text using one prompt template.

    Returns:
        raw_output: raw generated text from the model
        predicted_label: normalized predicted label
    """
    prompt = build_prompt(prompt_template, text, valid_labels)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    if model_type == "seq2seq":
        raw_output = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        ).strip()
    else:
        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        raw_output = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

    predicted_label = normalize_label(raw_output, valid_labels)
    return raw_output, predicted_label


def evaluate_prompt_predictions(
    y_true: List[str],
    y_pred: List[str]
) -> Dict[str, float]:
    """
    Calculate evaluation metrics for one prompt.
    """
    y_true_clean = [str(label).strip().lower() for label in y_true]
    y_pred_clean = [str(label).strip().lower() for label in y_pred]

    return {
        "accuracy": accuracy_score(y_true_clean, y_pred_clean),
        "f1_macro": f1_score(y_true_clean, y_pred_clean, average="macro", zero_division=0)
    }

def run_llm_prompt_evaluation(
    model_name: str,
    prompt_templates: Dict[str, str],   # <- DICT now
    max_input_length: int,
    max_new_tokens: int,
    input_df: pd.DataFrame,
    target_column_name: str,
    text_column_name: str,
    valid_labels: List[str],
    device: str
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run LLM-based classification for all rows and all prompt templates.

    Parameters:
        model_name:
            Hugging Face model name.
        prompt_templates:
            Dictionary of prompt templates. Each template must use:
            - {text}
            - {labels}
        max_input_length:
            Maximum number of input tokens.
        max_new_tokens:
            Maximum number of generated tokens.
        input_df:
            Input dataframe containing text and true labels.
        target_column_name:
            Column name containing true labels.
        text_column_name:
            Column name containing input text.
        valid_labels:
            List of valid class labels.
        device:
            'cpu' or 'cuda'.

    Returns:
        modified_input_df:
            Copy of input dataframe with added prediction columns:
            out_label_Prompt_1, out_label_Prompt_2, ...
        results_df:
            Dataframe with one row per prompt and evaluation metrics.
    """

    modified_input_df = input_df.copy()

    tokenizer, model, model_type = load_hf_model_and_tokenizer(
        model_name=model_name,
        device=device
    )

    y_true = modified_input_df[target_column_name].astype(str).str.lower().tolist()
    texts = modified_input_df[text_column_name].astype(str).tolist()

    results = []

    # Loop over dictionary
    for idx, (prompt_name, prompt_template) in enumerate(prompt_templates.items(), start=1):

        # Required column format (assignment requirement)
        column_name = f"out_label_Prompt_{idx}"

        predictions = []

        for text in texts:
            _, pred_label = predict_single_text(
                text=text,
                prompt_template=prompt_template,
                valid_labels=valid_labels,
                tokenizer=tokenizer,
                model=model,
                model_type=model_type,
                max_input_length=max_input_length,
                max_new_tokens=max_new_tokens,
                device=device
            )
            predictions.append(pred_label)

        # Add predictions to dataframe
        modified_input_df[column_name] = predictions

        # Evaluate
        scores = evaluate_prompt_predictions(y_true=y_true, y_pred=predictions)

        results.append({
            "model_name": model_name,
            "prompt_name": prompt_name,   # <- keep dictionary key
            "prediction_column": column_name,
            "accuracy": scores["accuracy"],
            "f1_macro": scores["f1_macro"]
        })

    results_df = pd.DataFrame(results)

    return modified_input_df, results_df

def save_results(predicted_df: pd.DataFrame, 
                 results_df: pd.DataFrame, 
                 prompt_templates: Dict[str, str], 
                 base_dir: str = 'data/outputs') -> None:
    """
    Save the predicted dataframe, results dataframe, and prompt templates 
    to a new timestamped folder.

    Args:
        predicted_df (pd.DataFrame): DataFrame containing predictions.
        results_df (pd.DataFrame): DataFrame containing evaluation metrics.
        prompt_templates (Dict[str, str]): Dictionary of prompt templates.
        base_dir (str): Base directory where results will be saved.
    """
    # Create a new folder with an incremented number
    existing_folders = [int(folder) for folder in os.listdir(base_dir) if folder.isdigit()]
    next_folder = str(max(existing_folders) + 1) if existing_folders else '1'
    new_folder_path = os.path.join(base_dir, next_folder)
    os.makedirs(new_folder_path)

    # Save the predicted dataframe
    predicted_df.to_csv(os.path.join(new_folder_path, 'predicted_df.csv'), index=False)

    # Save the results dataframe
    results_df.to_csv(os.path.join(new_folder_path, 'results_df.csv'), index=False)

    # Save the prompt templates as a CSV file
    prompt_templates_df = pd.DataFrame(list(prompt_templates.items()), columns=['Prompt Name', 'Prompt Template'])
    prompt_templates_df.to_csv(os.path.join(new_folder_path, 'prompt_templates.csv'), index=False)

Run for a prompt and LLM

In [7]:
# Load Data
valid_df = pd.read_csv('data/6/valid.csv')

# Configuration
PROMPT_TEMPLATES = {
    "prompt_1": """
    You are a text classification system.

    Classify the sentiment of the following text into exactly one label from:
    {labels}

    Return only one label.

    Text:
    {text}

    Label:
    """.strip(),

    "prompt_2": """
    Classify the sentiment of this text as one of these labels: {labels}.

    Text: {text}

    Answer with only one label.
    """.strip(),

    "prompt_3": """
    You must perform sentiment classification.

    Allowed labels:
    {labels}

    Rules:
    - Choose one label only
    - Do not explain

    Text:
    {text}

    Sentiment:
    """.strip(),
    }

In [ ]:
MODEL_NAME = "google/flan-t5-small"

In [8]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

cardiffnlp_df, cardiffnlp_results_df = run_llm_prompt_evaluation(
    model_name=MODEL_NAME,
    prompt_templates=PROMPT_TEMPLATES,
    max_input_length=512,
    max_new_tokens=10,
    input_df=valid_df,
    target_column_name="sentiment",
    text_column_name="text",
    valid_labels=["positive", "negative"],
    device="cpu"
)

cardiffnlp_results_df

If you want to use `RobertaLMHeadModel` as a standalone, add `is_decoder=True.`
Some weights of RobertaForCausalLM were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


IndexError: index out of range in self

Choosing LLMs

In [4]:
from __future__ import annotations

from typing import Dict, List, Tuple, Any
import pandas as pd


# -------------------------------------------------------------------
# Candidate models
# -------------------------------------------------------------------
CANDIDATE_MODELS: List[str] = [
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "distilbert-base-uncased-finetuned-sst-2-english",
    "siebert/sentiment-roberta-large-english",
    "finiteautomata/bertweet-base-sentiment-analysis",
    "nlptown/bert-base-multilingual-uncased-sentiment",
]


# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------
def _find_metric_column(results_df: pd.DataFrame, metric_keyword: str) -> str:
    """
    Find a metric column in results_df by keyword matching.

    Parameters
    ----------
    results_df : pd.DataFrame
        Evaluation results dataframe returned by run_llm_prompt_evaluation.
    metric_keyword : str
        Keyword to search for, e.g. "f1" or "accuracy".

    Returns
    -------
    str
        Matched column name.

    Raises
    ------
    ValueError
        If no matching column is found.
    """
    candidates = [
        col for col in results_df.columns
        if metric_keyword.lower() in col.lower()
    ]

    if not candidates:
        raise ValueError(
            f"Could not find a column containing '{metric_keyword}' in results_df. "
            f"Available columns: {list(results_df.columns)}"
        )

    # Prefer exact/common names first
    preferred_order = [
        metric_keyword,
        f"{metric_keyword}_score",
        f"macro_{metric_keyword}",
        f"weighted_{metric_keyword}",
        f"avg_{metric_keyword}",
    ]

    for preferred in preferred_order:
        for col in candidates:
            if col.lower() == preferred.lower():
                return col

    # Otherwise return the first match
    return candidates[0]


def _extract_prompt_level_metrics(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize prompt-level evaluation results.

    This function tries to extract:
    - prompt identifier column
    - f1 column
    - accuracy column

    Parameters
    ----------
    results_df : pd.DataFrame
        Output dataframe from run_llm_prompt_evaluation.

    Returns
    -------
    pd.DataFrame
        Standardized dataframe with columns:
        ['prompt', 'f1_score', 'accuracy']

    Raises
    ------
    ValueError
        If F1 or accuracy column cannot be found.
    """
    df = results_df.copy()

    # Detect prompt column
    prompt_candidates = [
        col for col in df.columns
        if "prompt" in col.lower()
    ]

    if prompt_candidates:
        prompt_col = prompt_candidates[0]
    else:
        # Fallback: use row index as prompt id
        df = df.reset_index(drop=False).rename(columns={"index": "prompt"})
        prompt_col = "prompt"

    f1_col = _find_metric_column(df, "f1")
    acc_col = _find_metric_column(df, "accuracy")

    standardized_df = df[[prompt_col, f1_col, acc_col]].copy()
    standardized_df.columns = ["prompt", "f1_score", "accuracy"]

    return standardized_df


# -------------------------------------------------------------------
# Main evaluation wrapper
# -------------------------------------------------------------------
def evaluate_models_across_prompts(
    candidate_models: List[str],
    prompt_templates: Dict[str, str],
    input_df: pd.DataFrame,
    target_column_name: str,
    text_column_name: str,
    valid_labels: List[str],
    device: str = "cpu",
    max_input_length: int = 512,
    max_new_tokens: int = 10,
) -> Tuple[pd.DataFrame, Dict[str, Dict[str, pd.DataFrame]]]:
    """
    Evaluate multiple Hugging Face models using run_llm_prompt_evaluation,
    then rank them by average F1-score across prompts.

    Assumes run_llm_prompt_evaluation returns:
        modified_input_df, results_df

    Parameters
    ----------
    candidate_models : List[str]
        List of model names to evaluate.
    prompt_templates : Dict[str, str]
        Prompt templates dictionary, e.g.:
        {
            "prompt_1": "...",
            "prompt_2": "...",
            "prompt_3": "..."
        }
    input_df : pd.DataFrame
        Validation dataframe.
    target_column_name : str
        Name of target/label column.
    text_column_name : str
        Name of text column.
    valid_labels : List[str]
        Allowed prediction labels.
    device : str, default="cpu"
        Device passed to run_llm_prompt_evaluation.
    max_input_length : int, default=512
        Max input length passed to run_llm_prompt_evaluation.
    max_new_tokens : int, default=10
        Max new tokens passed to run_llm_prompt_evaluation.

    Returns
    -------
    leaderboard_df : pd.DataFrame
        Sorted dataframe of models with average F1 and accuracy.
    all_outputs : Dict[str, Dict[str, pd.DataFrame]]
        Raw outputs for each model:
        {
            model_name: {
                "predictions_df": ...,
                "results_df": ...,
                "prompt_metrics_df": ...
            }
        }
    """
    leaderboard_rows: List[Dict[str, Any]] = []
    all_outputs: Dict[str, Dict[str, pd.DataFrame]] = {}

    for model_name in candidate_models:
        print(f"Evaluating model: {model_name}")

        try:
            predictions_df, results_df = run_llm_prompt_evaluation(
                model_name=model_name,
                prompt_templates=prompt_templates,
                max_input_length=max_input_length,
                max_new_tokens=max_new_tokens,
                input_df=input_df,
                target_column_name=target_column_name,
                text_column_name=text_column_name,
                valid_labels=valid_labels,
                device=device,
            )

            prompt_metrics_df = _extract_prompt_level_metrics(results_df)

            avg_f1 = prompt_metrics_df["f1_score"].mean()
            avg_accuracy = prompt_metrics_df["accuracy"].mean()

            leaderboard_rows.append({
                "model_name": model_name,
                "model_family": (
                    "RoBERTa" if "roberta" in model_name.lower()
                    else "DistilBERT" if "distilbert" in model_name.lower()
                    else "BERTweet" if "bertweet" in model_name.lower()
                    else "BERT" if "bert" in model_name.lower()
                    else "Other"
                ),
                "num_prompts_evaluated": len(prompt_metrics_df),
                "avg_f1_score": avg_f1,
                "avg_accuracy": avg_accuracy,
            })

            all_outputs[model_name] = {
                "predictions_df": predictions_df,
                "results_df": results_df,
                "prompt_metrics_df": prompt_metrics_df,
            }

        except Exception as e:
            print(f"Failed for model '{model_name}': {e}")
            leaderboard_rows.append({
                "model_name": model_name,
                "model_family": (
                    "RoBERTa" if "roberta" in model_name.lower()
                    else "DistilBERT" if "distilbert" in model_name.lower()
                    else "BERTweet" if "bertweet" in model_name.lower()
                    else "BERT" if "bert" in model_name.lower()
                    else "Other"
                ),
                "num_prompts_evaluated": 0,
                "avg_f1_score": None,
                "avg_accuracy": None,
            })

    leaderboard_df = pd.DataFrame(leaderboard_rows)

    # Sort by average F1-score first, then average accuracy
    leaderboard_df = leaderboard_df.sort_values(
        by=["avg_f1_score", "avg_accuracy"],
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)

    return leaderboard_df, all_outputs

In [5]:
leaderboard_df, all_outputs = evaluate_models_across_prompts(
    candidate_models=CANDIDATE_MODELS,
    prompt_templates=PROMPT_TEMPLATES,
    input_df=valid_df,
    target_column_name="sentiment",
    text_column_name="text",
    valid_labels=["positive", "negative"],
    device="cpu",
    max_input_length=512,
    max_new_tokens=10,
)

print("\n=== Model Leaderboard (sorted by average F1 across prompts) ===")
print(leaderboard_df)

Evaluating model: cardiffnlp/twitter-roberta-base-sentiment-latest


`torch_dtype` is deprecated! Use `dtype` instead!
If you want to use `RobertaLMHeadModel` as a standalone, add `is_decoder=True.`
Some weights of RobertaForCausalLM were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (514). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


Failed for model 'cardiffnlp/twitter-roberta-base-sentiment-latest': index out of range in self
Evaluating model: distilbert-base-uncased-finetuned-sst-2-english
Failed for model 'distilbert-base-uncased-finetuned-sst-2-english': Unrecognized configuration class <class 'transformers.models.distilbert.configuration_distilbert.DistilBertConfig'> for this kind of AutoModel: AutoModelForCausalLM.
Model type should be one of ApertusConfig, ArceeConfig, AriaTextConfig, BambaConfig, BartConfig, BertConfig, BertGenerationConfig, BigBirdConfig, BigBirdPegasusConfig, BioGptConfig, BitNetConfig, BlenderbotConfig, BlenderbotSmallConfig, BloomConfig, BltConfig, CamembertConfig, LlamaConfig, CodeGenConfig, CohereConfig, Cohere2Config, CpmAntConfig, CTRLConfig, Data2VecTextConfig, DbrxConfig, DeepseekV2Config, DeepseekV3Config, DiffLlamaConfig, DogeConfig, Dots1Config, ElectraConfig, Emu3Config, ErnieConfig, Ernie4_5Config, Ernie4_5_MoeConfig, Exaone4Config, FalconConfig, FalconH1Config, FalconMambaC

If you want to use `RobertaLMHeadModel` as a standalone, add `is_decoder=True.`
Some weights of RobertaForCausalLM were not initialized from the model checkpoint at siebert/sentiment-roberta-large-english and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Failed for model 'siebert/sentiment-roberta-large-english': index out of range in self
Evaluating model: finiteautomata/bertweet-base-sentiment-analysis


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
If you want to use `RobertaLMHeadModel` as a standalone, add `is_decoder=True.`
Some weights of RobertaForCausalLM were not initialized from the model checkpoint at finiteautomata/bertweet-base-sentiment-analysis and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Failed for model 'finiteautomata/bertweet-base-sentiment-analysis': index out of range in self
Evaluating model: nlptown/bert-base-multilingual-uncased-sentiment


If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`
Some weights of BertLMHeadModel were not initialized from the model checkpoint at nlptown/bert-base-multilingual-uncased-sentiment and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (512). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


Failed for model 'nlptown/bert-base-multilingual-uncased-sentiment': index out of range in self

=== Model Leaderboard (sorted by average F1 across prompts) ===
                                         model_name model_family  \
0  cardiffnlp/twitter-roberta-base-sentiment-latest      RoBERTa   
1   distilbert-base-uncased-finetuned-sst-2-english   DistilBERT   
2           siebert/sentiment-roberta-large-english      RoBERTa   
3   finiteautomata/bertweet-base-sentiment-analysis     BERTweet   
4  nlptown/bert-base-multilingual-uncased-sentiment         BERT   

   num_prompts_evaluated avg_f1_score avg_accuracy  
0                      0         None         None  
1                      0         None         None  
2                      0         None         None  
3                      0         None         None  
4                      0         None         None  


In [12]:
valid_df.head(100)[['sentiment','text']].to_csv('data/outputs/sample_valid_data.csv', index=False)

In [13]:
from __future__ import annotations

from typing import Dict, List, Tuple, Any
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
from sklearn.metrics import accuracy_score, f1_score


CANDIDATE_MODELS = [
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "distilbert-base-uncased-finetuned-sst-2-english",
    "siebert/sentiment-roberta-large-english",
    "finiteautomata/bertweet-base-sentiment-analysis",
    "nlptown/bert-base-multilingual-uncased-sentiment",
]


def _normalize_binary_label(text: str, valid_labels: List[str]) -> str | None:
    if text is None:
        return None

    t = str(text).strip().lower()

    # direct match
    if t in valid_labels:
        return t

    # common mappings
    mapping = {
        "label_0": "negative",
        "label_1": "positive",
        "neg": "negative",
        "pos": "positive",
        "1 star": "negative",
        "2 stars": "negative",
        "3 stars": "positive",
        "4 stars": "positive",
        "5 stars": "positive",
    }

    return mapping.get(t, None)


def _get_model_family(model_name: str) -> str:
    name = model_name.lower()
    if "distilbert" in name:
        return "DistilBERT"
    if "bertweet" in name:
        return "BERTweet"
    if "roberta" in name:
        return "RoBERTa"
    if "bert" in name:
        return "BERT"
    return "Other"


def _estimate_param_count(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def evaluate_sequence_classifier(
    model_name: str,
    input_df: pd.DataFrame,
    target_column_name: str,
    text_column_name: str,
    valid_labels: List[str],
    max_input_length: int = 512,
    batch_size: int = 16,
    device: str = "cpu",
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Evaluate one Hugging Face sequence-classification model.
    """

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()

    if device == "mps" and torch.backends.mps.is_available():
        torch_device = torch.device("mps")
    elif device == "cuda" and torch.cuda.is_available():
        torch_device = torch.device("cuda")
    else:
        torch_device = torch.device("cpu")

    model.to(torch_device)

    texts = input_df[text_column_name].astype(str).tolist()
    y_true = input_df[target_column_name].astype(str).str.lower().tolist()

    predictions = []
    confidences = []

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]

            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_input_length,
                return_tensors="pt",
            )
            enc = {k: v.to(torch_device) for k, v in enc.items()}

            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=-1)
            pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()
            pred_scores = torch.max(probs, dim=-1).values.cpu().numpy()

            for pred_id, score in zip(pred_ids, pred_scores):
                raw_label = model.config.id2label[int(pred_id)]
                norm_label = _normalize_binary_label(raw_label, valid_labels)
                predictions.append(norm_label)
                confidences.append(float(score))

    result_df = input_df.copy()
    result_df["predicted_label"] = predictions
    result_df["confidence"] = confidences

    valid_mask = result_df["predicted_label"].isin(valid_labels)
    eval_df = result_df.loc[valid_mask].copy()

    metrics = {
        "model_name": model_name,
        "model_family": _get_model_family(model_name),
        "num_parameters": _estimate_param_count(model),
        "num_samples": len(result_df),
        "num_valid_predictions": len(eval_df),
        "accuracy": accuracy_score(
            eval_df[target_column_name].str.lower(),
            eval_df["predicted_label"],
        ) if len(eval_df) > 0 else None,
        "macro_f1": f1_score(
            eval_df[target_column_name].str.lower(),
            eval_df["predicted_label"],
            average="macro",
        ) if len(eval_df) > 0 else None,
        "weighted_f1": f1_score(
            eval_df[target_column_name].str.lower(),
            eval_df["predicted_label"],
            average="weighted",
        ) if len(eval_df) > 0 else None,
    }

    return result_df, metrics


def evaluate_top_sentiment_models(
    candidate_models: List[str],
    input_df: pd.DataFrame,
    target_column_name: str,
    text_column_name: str,
    valid_labels: List[str],
    max_input_length: int = 512,
    batch_size: int = 16,
    device: str = "cpu",
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    Evaluate multiple sequence-classification models and sort by macro F1.
    """

    leaderboard_rows = []
    all_predictions = {}

    for model_name in candidate_models:
        print(f"Evaluating: {model_name}")
        try:
            pred_df, metrics = evaluate_sequence_classifier(
                model_name=model_name,
                input_df=input_df,
                target_column_name=target_column_name,
                text_column_name=text_column_name,
                valid_labels=valid_labels,
                max_input_length=max_input_length,
                batch_size=batch_size,
                device=device,
            )
            leaderboard_rows.append(metrics)
            all_predictions[model_name] = pred_df
        except Exception as e:
            print(f"Failed: {model_name} -> {e}")
            leaderboard_rows.append({
                "model_name": model_name,
                "model_family": _get_model_family(model_name),
                "num_parameters": None,
                "num_samples": len(input_df),
                "num_valid_predictions": 0,
                "accuracy": None,
                "macro_f1": None,
                "weighted_f1": None,
            })

    leaderboard_df = pd.DataFrame(leaderboard_rows)
    leaderboard_df = leaderboard_df.sort_values(
        by=["macro_f1", "accuracy"],
        ascending=False,
        na_position="last",
    ).reset_index(drop=True)

    return leaderboard_df, all_predictions

In [14]:
leaderboard_df, all_predictions = evaluate_top_sentiment_models(
    candidate_models=CANDIDATE_MODELS,
    input_df=valid_df,
    target_column_name="sentiment",
    text_column_name="text",
    valid_labels=["positive", "negative"],
    max_input_length=512,
    batch_size=16,
    device="mps"  # use "mps" on Apple Silicon if available
)

print(leaderboard_df)

Evaluating: cardiffnlp/twitter-roberta-base-sentiment-latest


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Evaluating: distilbert-base-uncased-finetuned-sst-2-english
Evaluating: siebert/sentiment-roberta-large-english
Evaluating: finiteautomata/bertweet-base-sentiment-analysis


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Evaluating: nlptown/bert-base-multilingual-uncased-sentiment
                                         model_name model_family  \
0   finiteautomata/bertweet-base-sentiment-analysis     BERTweet   
1  cardiffnlp/twitter-roberta-base-sentiment-latest      RoBERTa   
2           siebert/sentiment-roberta-large-english      RoBERTa   
3  nlptown/bert-base-multilingual-uncased-sentiment         BERT   
4   distilbert-base-uncased-finetuned-sst-2-english   DistilBERT   

   num_parameters  num_samples  num_valid_predictions  accuracy  macro_f1  \
0       134902275          700                    578  0.955017  0.931641   
1       124647939          700                    600  0.946667  0.922719   
2       355361794          700                    700  0.935714  0.903367   
3       167360261          700                    700  0.921429  0.879473   
4        66955010          700                    700  0.790000  0.743897   

   weighted_f1  
0     0.955847  
1     0.947957  
2     0.936433  